# Etude pour afficher un résumé des logs.
Ne fonctionne qu'avec des fichiers bien structurés. Ne fonctionne qu'avec des dates en français (!)

In [150]:
# Simplifie le fichier de type ls -ltR sur les logs de Glims
# Idée : produire un fichier html  de synthèse.

from datetime import datetime, timedelta
import re
from pprint import  pprint

from IPython.display import display, HTML

In [151]:
def parse_ls_lr(contenu):
    resultats = {}
    repertoire_courant = None

    for ligne in contenu.splitlines():
        ligne = ligne.strip()

        # Détection d'un répertoire
        if ligne.startswith("./") and ligne.endswith(":"):
            repertoire_courant = ligne[:-1]  # enlever le ":"
            repertoire_courant = repertoire_courant[2:]
            resultats[repertoire_courant] = None
            continue

        # Détection d'un fichier
        if ligne.startswith("-") and repertoire_courant:
            # Exemple de date : "17 mars 18:40"
            match = re.search(r'(\d{1,2}) (\w+) +(\d{1,2}:\d{2})', ligne)
            if match:
                jour, mois, heure = match.groups()

                # convertir mois FR -> numéro
                mois_fr = {
                    "janv": 1, "févr": 2, "mars": 3, "avr": 4,
                    "mai": 5, "juin": 6, "juil": 7, "août": 8,
                    "sept": 9, "oct": 10, "nov": 11, "déc": 12
                }

                mois_num = mois_fr.get(mois[:4].lower())
                if mois_num is None:
                    continue

                date_obj = datetime(
                    year= datetime.now().year,
                    month=mois_num,
                    day=int(jour),
                    hour=int(heure.split(":")[0]),
                    minute=int(heure.split(":")[1])
                )

                # garder la plus récente
                if (resultats[repertoire_courant] is None or
                        date_obj > resultats[repertoire_courant]):
                    resultats[repertoire_courant] = date_obj

    return resultats


In [152]:
with open("logs/svc_logs.txt", 'r') as f:
        lines = f.read()
        result = parse_ls_lr(lines)

pprint(result)

{'JCE1': datetime.datetime(2026, 3, 17, 15, 42),
 'JCE2': datetime.datetime(2026, 3, 17, 16, 27),
 'JCE3': None,
 'cyberlabTestcron': datetime.datetime(2026, 3, 4, 10, 32),
 'cyberlabcron': datetime.datetime(2026, 3, 4, 10, 33),
 'cyberlabprodcron': datetime.datetime(2026, 3, 17, 18, 46),
 'glims_hexa_ident': datetime.datetime(2026, 3, 17, 18, 44),
 'glimscron1': datetime.datetime(2026, 3, 17, 15, 25),
 'glimscron2': datetime.datetime(2026, 3, 17, 18, 5),
 'glimscron3': datetime.datetime(2026, 3, 17, 18, 45),
 'glimscron4': datetime.datetime(2026, 3, 17, 18, 45),
 'glimscron5': datetime.datetime(2026, 3, 17, 18, 46),
 'glimscron6': datetime.datetime(2026, 3, 17, 18, 41),
 'glimscron7': datetime.datetime(2026, 3, 17, 18, 41),
 'glimscron8': datetime.datetime(2026, 3, 17, 18, 46),
 'glimscron_valab': datetime.datetime(2026, 3, 17, 18, 41),
 'glimsonl1': datetime.datetime(2026, 3, 17, 18, 45),
 'glimsonl10': datetime.datetime(2026, 3, 17, 18, 45),
 'glimsonl11': datetime.datetime(2026, 3,

In [153]:
len(result)

36

In [154]:
def filtrer_anciens(donnees, minutes=15):
    maintenant = datetime.now()
    seuil = timedelta(minutes=minutes)

    resultats = {}

    # for rep, date in donnees.items():
    for rep in sorted(donnees):
        date = donnees[rep]
        if date is None:
            continue

        age = maintenant - date

        if age > seuil:
            resultats[rep] = {
                "date": date,
                "age_minutes": int(age.total_seconds() // 60)
            }

    return resultats

In [155]:
def dict_to_html(data):
    html = []

    html.append("<table border='1' cellpadding='5' cellspacing='0'>")
    html.append("<tr><th>Répertoire</th><th>Dernière mise à jour</th><th>Âge (minutes)</th></tr>")

    for rep, info in data.items():
        date = info["date"]
        age = info["age_minutes"]

        # couleur si trop ancien
        couleur = "#ffcccc" if age > 53 else "#ccffcc"

        html.append(
            f"<tr style='background-color:{couleur};'>"
            f"<td>{rep}</td>"
            f"<td>{date.strftime('%d/%m/%Y %H:%M')}</td>"
            f"<td>{age}</td>"
            f"</tr>"
        )

    html.append("</table>")

    return "\n".join(html)

In [156]:
old = filtrer_anciens(result, minutes = 120)

In [157]:
HTML(dict_to_html(old))

Répertoire,Dernière mise à jour,Âge (minutes)
JCE1,17/03/2026 15:42,262
JCE2,17/03/2026 16:27,217
cyberlabTestcron,04/03/2026 10:32,19292
cyberlabcron,04/03/2026 10:33,19291
glimscron1,17/03/2026 15:25,279
glimsonl15,17/03/2026 12:33,451
glimsonl2,17/03/2026 17:12,172
glimsonl3,17/03/2026 10:36,568
glimsonl_valab,11/03/2026 07:31,9393


In [158]:
def filtre_dict(dico):
    for key in dico:
        val = dico()

In [159]:
old

{'JCE1': {'date': datetime.datetime(2026, 3, 17, 15, 42), 'age_minutes': 262},
 'JCE2': {'date': datetime.datetime(2026, 3, 17, 16, 27), 'age_minutes': 217},
 'cyberlabTestcron': {'date': datetime.datetime(2026, 3, 4, 10, 32),
  'age_minutes': 19292},
 'cyberlabcron': {'date': datetime.datetime(2026, 3, 4, 10, 33),
  'age_minutes': 19291},
 'glimscron1': {'date': datetime.datetime(2026, 3, 17, 15, 25),
  'age_minutes': 279},
 'glimsonl15': {'date': datetime.datetime(2026, 3, 17, 12, 33),
  'age_minutes': 451},
 'glimsonl2': {'date': datetime.datetime(2026, 3, 17, 17, 12),
  'age_minutes': 172},
 'glimsonl3': {'date': datetime.datetime(2026, 3, 17, 10, 36),
  'age_minutes': 568},
 'glimsonl_valab': {'date': datetime.datetime(2026, 3, 11, 7, 31),
  'age_minutes': 9393}}